# Figure Generation — ThermoRG-NN

Run on Kaggle (CPU). Generates all paper figures from data.
Output: /kaggle/working/figures/

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.optimize import curve_fit
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
import json, os, sys

sys.path.insert(0, '/kaggle/working/ThermoRG-NN')

OUT = '/kaggle/working/figures'
os.makedirs(OUT, exist_ok=True)
print(f'Output: {OUT}')
print(f'matplotlib: {matplotlib.__version__}')

---
## Figure 1: J_topo Universality

In [ ]:
import csv

phase_a = []
with open('/kaggle/working/ThermoRG-NN/phase_a_summary.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        phase_a.append(row)

families = {}
for r in phase_a:
    g = r['group']
    families.setdefault(g, []).append({
        'name': r['name'],
        'J': float(r['J_topo']),
        'alpha': float(r['alpha']),
        'beta': float(r.get('beta', 0)),
        'params': float(r['params_M'])
    })

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors = {'G1': 'C0', 'G2': 'C1', 'G3': 'C2', 'G4': 'C3'}
markers = {'G1': 'o', 'G2': 's', 'G3': '^', 'G4': 'D'}

# Panel A: J_topo vs alpha
for g, archs in families.items():
    xs = [a['alpha'] for a in archs]
    ys = [a['J'] for a in archs]
    axes[0].scatter(xs, ys, c=colors[g], marker=markers[g],
                   label=g, zorder=5)
axes[0].set_xlabel(r'alpha')
axes[0].set_ylabel(r'J_topo')
axes[0].set_title('(a) J_topo vs alpha')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel B: J_topo by family
for g, archs in families.items():
    ys = [a['J'] for a in archs]
    xs = [g] * len(ys)
    axes[1].scatter(xs, ys, c=colors[g], marker=markers[g],
                   label=g, zorder=5)
axes[1].set_xlabel('Family')
axes[1].set_ylabel(r'J_topo')
axes[1].set_title('(b) J_topo by family')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT}/fig1_J_topo_universality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig1 saved')

---
## Figure 4: J_topo(D) Log-Log Scaling

In [ ]:
data_L3 = {
    'D': np.array([16, 24, 32, 48, 64, 96]),
    'J': np.array([0.8429, 0.8126, 0.7807, 0.7351, 0.6944, 0.6472]),
    'std': np.array([0.016, 0.016, 0.013, 0.015, 0.007, 0.009])
}
data_L5 = {
    'D': np.array([16, 24, 32, 48, 64, 96]),
    'J': np.array([0.8684, 0.8653, 0.8442, 0.8149, 0.7768, 0.7515]),
    'std': np.array([0.01]*6)
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, d, L in zip(axes, [data_L3, data_L5], [3, 5]):
    D, J, std = d['D'], d['J'], d['std']
    log_D = np.log(D)
    log_J = np.log(J)
    slope, intercept = np.polyfit(log_D, log_J, 1)

    ax.errorbar(D, J, yerr=std, fmt='o', capsize=3, zorder=5)
    D_fit = np.linspace(12, 120, 200)
    J_fit = np.exp(intercept) * D_fit**slope
    ax.plot(D_fit, J_fit, '--', color='gray', label=f'fit: slope={slope:.3f}')
    ax.axhline(1.0, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$D$')
    ax.set_ylabel(r'J_topo')
    ax.set_title(f'L={L}: slope={slope:.3f}')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xlim(12, 120)

plt.suptitle(r'$J_{\mathrm{topo}}(D) \propto D^{-\ alpha_{\mathrm{GELU}}/L}$', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT}/fig4_J_topo_D_scaling.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig4 saved')

---
## Figure 5: beta(gamma) Cooling Theory

In [ ]:
gamma_vals = np.array([3.39, 2.29, 0.41])
beta_vals  = np.array([1.117, 0.950, 0.219])
norms = ['None', 'BN', 'LN']
cols = ['red', 'orange', 'blue']

g_curve = np.linspace(0.3, 4.0, 300)
b_curve = 0.425 * np.log(g_curve / 2.0) + 0.893

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(g_curve, b_curve, 'k--', alpha=0.5,
        label=r'$\beta(\gamma) = 0.425\ln(\gamma/2.0) + 0.893$')

for g, b, n, c in zip(gamma_vals, beta_vals, norms, cols):
    ax.scatter(g, b, s=100, color=c, zorder=5)
    ax.annotate(n, (g, b), xytext=(8, 0),
                textcoords='offset points', fontsize=11)

ax.axvline(2.0, color='gray', linestyle=':', alpha=0.7,
           label=r'$\gamma_c = 2.0$')
ax.set_xlabel(r'gamma')
ax.set_ylabel(r'beta')
ax.set_title('Cooling Theory: $\beta(\gamma)$ Validated')
ax.legend(fontsize=9, loc='upper right')
ax.grid(alpha=0.3)
ax.set_xlim(0, 4)
ax.set_ylim(0, 1.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig5_beta_gamma_cooling.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig5 saved')

---
## Figure 9: LN Scaling Law

In [ ]:
D_LN = np.array([32, 32, 48, 48, 64, 64, 96, 96])
loss_LN = np.array([1.0958, 1.1056, 1.0212, 1.0274, 0.9626, 0.9757, 0.9035, 0.9045])
D_unique = np.array([32, 48, 64, 96])
loss_avg = np.array([loss_LN[D_LN==d].mean() for d in D_unique])

def plaw(D, a, b, c): return a * D**(-b) + c
popt, _ = curve_fit(plaw, D_unique, loss_avg,
                     p0=[10, 0.5, 0.5],
                     bounds=([0,0,0],[1000,3,3]), maxfev=10000)
a, beta_LN, c = popt
loss_pred = plaw(D_unique, *popt)
r2 = 1 - (((loss_avg - loss_pred)**2).sum()) / (((loss_avg - loss_avg.mean())**2).sum())

D_fit = np.linspace(20, 120, 200)
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(D_unique, loss_avg, s=80, zorder=5,
           label='LN data (avg over 2 seeds)')
ax.plot(D_fit, plaw(D_fit, *popt), 'b-',
        label=f'fit: beta={beta_LN:.3f}, $R^2$={r2:.4f}')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$D$')
ax.set_ylabel('Val Loss')
ax.set_title('LN Cooling Validation (Phase B1)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig9_LN_scaling_law.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Fig9 saved: beta={beta_LN:.3f}, R2={r2:.4f}')

---
## Figure 10: HBO vs Random (Negative Result)

In [ ]:
random_l2 = [
    (0.3858, 0.7739, 96),
    (0.5014, 0.8062, 64),
    (0.6047, 0.7838, 64),
    (0.6268, 0.7538, 64),
    (0.6937, 0.8027, 48),
]
hbo_l2 = [
    (0.6051, 0.8627, 32),
    (0.6812, 0.8774, 24),
    (0.7479, 0.8455, 32),
    (0.7821, 0.8727, 24),
    (0.8378, 0.8701, 24),
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Panel A: bar chart
r_losses = [r[0] for r in random_l2]
h_losses = [r[0] for r in hbo_l2]
x = np.arange(5)
w = 0.35
axes[0].bar(x - w/2, r_losses, w, label='Random', color='steelblue')
axes[0].bar(x + w/2, h_losses, w, label='HBO', color='coral')
axes[0].set_xlabel('Rank')
axes[0].set_ylabel('Val Loss (50 epochs)')
axes[0].set_title('Random vs HBO')
axes[0].legend()
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'#{i+1}' for i in range(5)])
axes[0].grid(alpha=0.3, axis='y')

# Panel B: width distribution
r_widths = [r[2] for r in random_l2]
h_widths = [r[2] for r in hbo_l2]
axes[1].scatter([0]*len(r_widths), r_widths, s=80,
               c='steelblue', label='Random', zorder=5)
axes[1].scatter([1]*len(h_widths), h_widths, s=80,
               c='coral', label='HBO', zorder=5)
axes[1].set_xticks([0,1])
axes[1].set_xticklabels(['Random', 'HBO'])
axes[1].set_ylabel('Width')
axes[1].set_title('HBO Selected Narrow-Deep')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUT}/fig10_HBO_vs_Random.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig10 saved')

---
## Figure 12: Confounding Analysis

In [ ]:
phase_b2 = np.array([
    [96, 6, 0.7739, 0.3858],
    [64, 5, 0.8062, 0.5014],
    [64, 5, 0.7838, 0.6047],
    [64, 4, 0.7538, 0.6268],
    [48, 4, 0.8027, 0.6937],
    [32, 6, 0.8627, 0.6051],
    [24, 6, 0.8774, 0.6812],
    [32, 5, 0.8455, 0.7479],
    [24, 6, 0.8727, 0.7821],
    [24, 5, 0.8701, 0.8378],
])
widths = phase_b2[:, 0]
j_topo = phase_b2[:, 2]
losses = phase_b2[:, 3]

r_JL, p_JL = spearmanr(j_topo, losses)
r_WL, p_WL = spearmanr(widths, losses)
r_WJ, _   = spearmanr(widths, j_topo)

def partial_corr(x, y, z):
    lr_xz = LinearRegression().fit(z.reshape(-1,1), x)
    x_r = x - lr_xz.predict(z.reshape(-1,1))
    lr_yz = LinearRegression().fit(z.reshape(-1,1), y)
    y_r = y - lr_yz.predict(z.reshape(-1,1))
    return spearmanr(x_r, y_r)

r_JL_W, p_JL_W = partial_corr(j_topo, losses, widths)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Panel 1: J_topo vs Loss (colored by width)
sc1 = axes[0].scatter(j_topo, losses, c=widths, cmap='viridis',
                        s=80, zorder=5)
axes[0].set_xlabel(r'J_topo')
axes[0].set_ylabel('Val Loss')
axes[0].set_title(f'$J_{{\mathrm{topo}}}$ vs Loss\nSimple r={r_JL:+.3f}')
fig.colorbar(sc1, ax=axes[0], label='Width')
axes[0].grid(alpha=0.3)

# Panel 2: Width vs Loss (colored by J_topo)
sc2 = axes[1].scatter(widths, losses, c=j_topo, cmap='plasma',
                        s=80, zorder=5)
axes[1].set_xlabel('Width')
axes[1].set_ylabel('Val Loss')
axes[1].set_title(f'Width vs Loss\nSimple r={r_WL:+.3f}')
fig.colorbar(sc2, ax=axes[1], label='J_topo')
axes[1].grid(alpha=0.3)

# Panel 3: Width vs J_topo (colored by Loss)
sc3 = axes[2].scatter(widths, j_topo, c=losses, cmap='RdYlGn_r',
                        s=80, zorder=5)
axes[2].set_xlabel('Width')
axes[2].set_ylabel(r'J_topo')
axes[2].set_title(f'Width vs $J_{{\mathrm{topo}}}$\nConfounder r={r_WJ:+.3f}')
fig.colorbar(sc3, ax=axes[2], label='Loss')
axes[2].grid(alpha=0.3)

plt.suptitle("Simpson's Paradox: Confounding Analysis", fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT}/fig12_confounding.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig12 saved')

---
## Figure 13: Partial Correlation

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

labels = ['Simple\n$r(J,L)$', 'Partial\n$r(J,L|W)$']
values = [r_JL, r_JL_W]
ps = [p_JL, p_JL_W]
cols = ['#aaaaaa', 'steelblue']

bars = ax.bar(labels, values, color=cols, width=0.5)
ax.axhline(0, color='black', linewidth=0.5)

for bar, v, p in zip(bars, values, ps):
    ypos = v + 0.05 if v > 0 else v - 0.10
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'r={v:+.3f} (p={p:.3f})',
            ha='center', fontsize=11)

ax.set_ylabel(r'Spearman r')
ax.set_title(r'$J_{\mathrm{topo}} \to$ Loss\n(Controlling for Width)')
ax.set_ylim(-1.1, 1.1)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUT}/fig13_partial_corr.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Fig13 saved: r_JL={r_JL:.3f}, r_JL_W={r_JL_W:.3f}')